[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_2_opencv_foundations/05_image_operations.ipynb)

# Notebook 05 — Image Operations
### 6D Pose Vision Workshop · Part 2: OpenCV Foundations

**Estimated time:** 50 minutes  
**Dependencies:** `opencv-contrib-python`, `numpy`, `matplotlib`

---

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install opencv-python opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab — packages installed")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"OpenCV version: {cv2.__version__}")

# ── Unified display helper (same as previous notebook) ───────────────────────
def show_images(pairs, figsize_per=(5, 4)):
    """Display a list of (image, title) pairs side by side."""
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0] * n, figsize_per[1]))
    if n == 1:
        axes = [axes]
    for ax, (img, title) in zip(axes, pairs):
        if img.ndim == 3:
            ax.imshow(img[:, :, ::-1])  # BGR → RGB
        else:
            ax.imshow(img, cmap='gray')
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("Helper functions loaded.")

In [ ]:
# Create test images for this notebook

os.makedirs('../assets/images', exist_ok=True)

def create_industrial_scene(h=480, w=640):
    """Synthetic industrial scene with various textures for processing demos."""
    img = np.full((h, w, 3), [180, 180, 175], dtype=np.uint8)  # concrete floor
    
    # Metal box / pallet
    cv2.rectangle(img, (80, 200), (350, 420), (80, 90, 100), -1)
    cv2.rectangle(img, (80, 200), (350, 420), (40, 40, 40), 3)
    # Pallet fork openings
    cv2.rectangle(img, (110, 320), (200, 380), (30, 30, 30), -1)
    cv2.rectangle(img, (230, 320), (320, 380), (30, 30, 30), -1)
    
    # Colored barrel (orange — good for HSV segmentation)
    cv2.ellipse(img, (500, 350), (70, 100), 0, 0, 360, (20, 100, 230), -1)
    cv2.ellipse(img, (500, 350), (70, 100), 0, 0, 360, (0, 60, 180), 3)
    # Barrel bands
    cv2.ellipse(img, (500, 295), (70, 15), 0, 0, 360, (0, 60, 180), 3)
    cv2.ellipse(img, (500, 405), (70, 15), 0, 0, 360, (0, 60, 180), 3)
    
    # Grid lines on floor (adds texture for edge detection)
    for x in range(0, w, 60):
        cv2.line(img, (x, 0), (x, h), (160, 160, 155), 1)
    for y in range(0, h, 60):
        cv2.line(img, (0, y), (w, y), (160, 160, 155), 1)
    
    # Add noise to simulate camera sensor noise
    noise = np.random.normal(0, 8, img.shape).astype(np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    return img

scene = create_industrial_scene()
cv2.imwrite('../assets/images/industrial_scene.jpg', scene)

show_images([(scene, 'Industrial Scene (our test image)')])
print(f"Image shape: {scene.shape}")

## Learning Objectives

By the end of this notebook you will:

- Resize images with the right interpolation method for the task
- Crop a region of interest (ROI) using array slicing
- Convert between BGR, Grayscale, HSV, and LAB color spaces
- Apply thresholding (global, adaptive, Otsu's)
- Use morphological operations (erosion, dilation, opening, closing)
- Blur images with Gaussian, Median, and Bilateral filters
- Detect edges with Sobel and Canny
- Build a practical image preprocessing pipeline

---

## 1. Resize and Interpolation

### cv2.resize

```python
cv2.resize(src, dsize, interpolation=cv2.INTER_LINEAR)
# dsize = (width, height) — note: OpenCV is (W, H), NumPy is (H, W)
```

### Interpolation methods — when to use which

| Method | Use when | Quality | Speed |
|---|---|---|---|
| `INTER_NEAREST` | Speed is critical, blocky OK | Low | Fastest |
| `INTER_LINEAR` | General upsampling (default) | Good | Fast |
| `INTER_AREA` | **Downsampling** — avoids aliasing | Best for shrink | Medium |
| `INTER_CUBIC` | High quality upsampling | High | Slow |
| `INTER_LANCZOS4` | Maximum quality upsampling | Highest | Slowest |

In [ ]:
img = scene.copy()
h, w = img.shape[:2]
print(f"Original: {w}×{h}")

# Resize to specific dimensions (W, H order!)
img_small = cv2.resize(img, (320, 240))  # half size
print(f"Half size: {img_small.shape[1]}×{img_small.shape[0]}")

# Resize by scale factor
scale = 0.5
img_scaled = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
print(f"50% scale (INTER_AREA): {img_scaled.shape[1]}×{img_scaled.shape[0]}")

# Compare interpolation methods on upsampling a small patch
patch = img[200:260, 80:140]  # 60×60 crop
target_size = (240, 240)

methods = [
    (cv2.INTER_NEAREST, 'NEAREST\n(blocky)'),
    (cv2.INTER_LINEAR,  'LINEAR\n(default)'),
    (cv2.INTER_CUBIC,   'CUBIC\n(smooth)'),
]

resized = [(cv2.resize(patch, target_size, interpolation=m), name)
           for m, name in methods]
show_images(resized)
print("Upsampled 60×60 → 240×240 patch with different interpolation methods")

In [ ]:
# Cropping (Region of Interest) — just NumPy slicing
# Syntax: img[y_start:y_end, x_start:x_end]
# In NumPy:  row (y) is first, col (x) is second

img = scene.copy()

# Extract the pallet region
pallet_roi = img[200:420, 80:350]
print(f"Pallet ROI shape: {pallet_roi.shape}")

# Extract the barrel region
barrel_roi = img[250:450, 430:570]

# Annotate the original to show ROI boundaries
annotated = img.copy()
cv2.rectangle(annotated, (80, 200), (350, 420), (0, 255, 0), 2)
cv2.putText(annotated, 'Pallet ROI', (85, 195),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
cv2.rectangle(annotated, (430, 250), (570, 450), (0, 200, 255), 2)
cv2.putText(annotated, 'Barrel ROI', (435, 245),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2)

show_images([
    (annotated, 'ROIs marked on original'),
    (pallet_roi, f'Pallet ROI {pallet_roi.shape[:2]}'),
    (barrel_roi, f'Barrel ROI {barrel_roi.shape[:2]}')
])

## 2. Color Space Conversions

### Why different color spaces?

**BGR** mixes color and brightness together — changing the "redness" of a pixel also changes its apparent brightness. Other color spaces separate them:

| Color Space | Channels | Best for |
|---|---|---|
| BGR | Blue, Green, Red | OpenCV default; image display |
| Grayscale | Intensity only | Edge detection, thresholding, performance |
| HSV | Hue, Saturation, Value | **Color-based object segmentation** (best!) |
| LAB | Lightness, A (green-red), B (blue-yellow) | Perceptually uniform; face detection, matching |
| YCrCb | Y (luma), Cr (red diff), Cb (blue diff) | Skin tone detection, JPEG compression |

**HSV is the most useful for robotics** because you can segment objects by color without lighting sensitivity — a red barrel is still "red" (hue ≈ 0°) whether you're in bright or dim light.

### OpenCV HSV ranges

```
H (Hue):        0–179  (OpenCV uses half-scale; multiply by 2 for standard 0–360°)
S (Saturation): 0–255  (0=gray, 255=full color)
V (Value):      0–255  (0=black, 255=bright)
```

Common hue ranges (OpenCV scale):
```
Red:    0–10 and 170–179 (wraps around!)
Orange: 10–25
Yellow: 25–35
Green:  35–85
Cyan:   85–100
Blue:   100–130
Purple: 130–155
```

In [ ]:
img = scene.copy()

# Convert to different color spaces
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

# Show each channel of HSV
show_images([
    (img,      'BGR Original'),
    (gray,     'Grayscale'),
    (hsv[:,:,0], 'HSV: Hue'),
    (hsv[:,:,1], 'HSV: Saturation'),
    (hsv[:,:,2], 'HSV: Value (brightness)'),
])

# The orange barrel has high saturation — notice it stands out in the S channel

In [ ]:
# HSV color segmentation — isolate the orange barrel
# This is a core technique for colored object detection in robotics

hsv = cv2.cvtColor(scene, cv2.COLOR_BGR2HSV)

# Define orange range in HSV (OpenCV scale: H: 0-179, S: 0-255, V: 0-255)
lower_orange = np.array([10, 120, 100])   # H=10 (orange start), S and V minimums
upper_orange = np.array([30, 255, 255])   # H=30 (orange end)

# Create binary mask: 255 where color matches, 0 elsewhere
mask = cv2.inRange(hsv, lower_orange, upper_orange)

# Apply mask to original image
segmented = cv2.bitwise_and(scene, scene, mask=mask)

show_images([
    (scene,     'Original'),
    (mask,      'Orange Mask'),
    (segmented, 'Segmented Barrel'),
])

pixel_count = np.sum(mask > 0)
print(f"Orange pixels detected: {pixel_count} ({pixel_count / mask.size * 100:.1f}% of image)")

## 3. Thresholding

### The concept

Thresholding converts a grayscale image to binary (black/white): pixels above a threshold become 255, below become 0 (or vice versa).

$$\text{dst}(x,y) = \begin{cases} 255 & \text{if } \text{src}(x,y) > T \\ 0 & \text{otherwise} \end{cases}$$

Where $T$ is the threshold value.

### Three methods

| Method | When to use |
|---|---|
| Global | Uniform lighting, clear foreground/background separation |
| Adaptive | Variable lighting — threshold is computed locally per region |
| Otsu's | Automatically finds the optimal global threshold |

**For robotics:** Otsu's + preprocessing (blur first) is usually the most robust global method. Adaptive thresholding handles shadows and uneven illumination.

In [ ]:
gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)

# 1. Global threshold — fixed value T=127
_, thresh_global = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# 2. Otsu's threshold — automatically finds optimal T
# cv2.threshold with THRESH_OTSU returns (optimal_T, binary_image)
optimal_T, thresh_otsu = cv2.threshold(
    gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)
print(f"Otsu's optimal threshold: {optimal_T:.0f}")

# 3. Adaptive threshold — uses local neighbourhood
thresh_adaptive = cv2.adaptiveThreshold(
    gray,
    maxValue=255,
    adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    thresholdType=cv2.THRESH_BINARY,
    blockSize=31,   # size of neighbourhood (must be odd)
    C=5             # constant subtracted from neighbourhood mean
)

show_images([
    (gray,           'Grayscale'),
    (thresh_global,  f'Global T=127'),
    (thresh_otsu,    f"Otsu's T={optimal_T:.0f}"),
    (thresh_adaptive,'Adaptive (local)'),
])

# Tip: blur before Otsu for noisy images
gray_blurred = cv2.GaussianBlur(gray, (5, 5), 0)
T_clean, thresh_clean = cv2.threshold(
    gray_blurred, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)
show_images([
    (thresh_otsu,  f"Otsu without blur (T={optimal_T:.0f})"),
    (thresh_clean, f"Otsu after Gaussian blur (T={T_clean:.0f})"),
])

## 4. Morphological Operations

### The concept

Morphological operations modify the **shape** of binary regions by sliding a **structuring element** (kernel) over the image.

| Operation | Effect | Use for |
|---|---|---|
| **Erosion** | Shrinks white regions, removes small noise | Remove tiny islands of noise |
| **Dilation** | Expands white regions, fills small holes | Fill small gaps in detected regions |
| **Opening** | Erosion then Dilation | Remove noise while preserving shape |
| **Closing** | Dilation then Erosion | Fill holes while preserving shape |

**Opening** and **Closing** are the most useful in practice:
- **Opening** = clean up noise in a binary mask
- **Closing** = fill small gaps in detected objects

In [ ]:
# Get a binary mask to demonstrate morphological ops
hsv = cv2.cvtColor(scene, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, np.array([10, 120, 100]), np.array([30, 255, 255]))

# Create the structuring element (kernel)
kernel_3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
kernel_7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

# Apply operations
eroded   = cv2.erode(mask, kernel_3, iterations=2)
dilated  = cv2.dilate(mask, kernel_3, iterations=2)
opened   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel_7)  # remove noise
closed   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_7)  # fill gaps

show_images([
    (mask,    'Original mask'),
    (eroded,  'Eroded (shrinks)'),
    (dilated, 'Dilated (expands)'),
    (opened,  'Opened (noise removed)'),
    (closed,  'Closed (gaps filled)'),
])

print("Typical robotics workflow:")
print("  1. HSV threshold → rough mask")
print("  2. Open → remove small noise pixels")
print("  3. Close → fill holes in detected object")
print("  4. findContours → find object boundary")

## 5. Blurring and Smoothing

### Why blur before processing?

Real camera images have **sensor noise** — random pixel value fluctuations. Edge detectors, thresholds, and feature detectors are sensitive to noise. Blurring first removes noise and makes subsequent operations more stable.

### Three filters

| Filter | How it works | Best for |
|---|---|---|
| **Gaussian** | Weighted average (center pixels weighted more) | General smoothing — most common |
| **Median** | Replaces pixel with median of neighbourhood | Remove salt-and-pepper noise, preserve edges |
| **Bilateral** | Like Gaussian but edge-aware (preserves edges) | Smooth texture while keeping object boundaries |

**Rule of thumb for robotics:**
- Before edge detection: Gaussian blur (fast)
- Before thresholding noisy cameras: Median blur (preserves edges better)
- Before feature matching where sharpness matters: Bilateral

In [ ]:
# Add salt-and-pepper noise to demonstrate blur comparison
noisy = scene.copy().astype(np.int16)
noise = np.random.randint(-40, 40, scene.shape, dtype=np.int16)
noisy = np.clip(noisy + noise, 0, 255).astype(np.uint8)

# Apply three blur types
blur_gaussian  = cv2.GaussianBlur(noisy, (7, 7), sigmaX=0)
blur_median    = cv2.medianBlur(noisy, 7)        # kernel must be odd
blur_bilateral = cv2.bilateralFilter(noisy,
                                     d=9,         # neighbourhood diameter
                                     sigmaColor=75,  # color similarity range
                                     sigmaSpace=75)  # spatial range

show_images([
    (scene,          'Original (clean)'),
    (noisy,          'Noisy'),
    (blur_gaussian,  'Gaussian (7×7)'),
    (blur_median,    'Median (7×7)'),
    (blur_bilateral, 'Bilateral (d=9)'),
])

print("Bilateral filter preserves the sharp edge of the pallet — notice the cleaner outline vs Gaussian")

## 6. Edge Detection — Sobel and Canny

### The concept

Edges are places where pixel intensity changes rapidly. The rate of change is the **gradient** — a vector pointing in the direction of steepest intensity increase.

$$G_x = \frac{\partial I}{\partial x} \quad G_y = \frac{\partial I}{\partial y} \quad G = \sqrt{G_x^2 + G_y^2}$$

Where:
- $G_x$ = horizontal gradient (detects vertical edges)
- $G_y$ = vertical gradient (detects horizontal edges)
- $G$ = gradient magnitude (all edges)

### Sobel — gradient-based edges

Sobel applies a small kernel that approximates the gradient. It gives a **gradient magnitude image** — bright where edges are strong.

### Canny — the gold standard

Canny is a multi-stage algorithm:
1. **Gaussian blur** — reduce noise
2. **Sobel gradient** — find edge magnitudes
3. **Non-maximum suppression** — thin edges to 1 pixel wide
4. **Hysteresis thresholding** — keep strong edges, keep weak edges connected to strong ones

Canny produces clean, thin, connected edges. Use it when you need precise object boundaries.

In [ ]:
gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
gray_blurred = cv2.GaussianBlur(gray, (3, 3), 0)

# Sobel edges
sobel_x = cv2.Sobel(gray_blurred, cv2.CV_64F, 1, 0, ksize=3)  # horizontal gradient
sobel_y = cv2.Sobel(gray_blurred, cv2.CV_64F, 0, 1, ksize=3)  # vertical gradient
sobel_magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
sobel_uint8 = np.clip(sobel_magnitude, 0, 255).astype(np.uint8)

# Sobel direction (for orientation-aware processing)
sobel_direction = np.arctan2(sobel_y, sobel_x) * 180 / np.pi

# Canny edges — two thresholds control hysteresis
canny_strict = cv2.Canny(gray_blurred, threshold1=50,  threshold2=150)
canny_loose  = cv2.Canny(gray_blurred, threshold1=20,  threshold2=80)

show_images([
    (gray,          'Grayscale'),
    (sobel_uint8,   'Sobel magnitude'),
    (canny_strict,  'Canny (strict: 50, 150)'),
    (canny_loose,   'Canny (loose: 20, 80)'),
])

print("Canny threshold tips:")
print("  - threshold1 (low):  edges below this are discarded")
print("  - threshold2 (high): edges above this are kept")
print("  - edges between thresholds are kept only if connected to strong edges")
print("  - Ratio 1:3 (e.g., 50:150) is a common starting point")

In [ ]:
# Finding contours from edges — the full pipeline
# This is how we detect object shapes in robotics

edges = cv2.Canny(gray_blurred, 40, 120)

# Find contours — boundaries of white regions in binary image
contours, hierarchy = cv2.findContours(
    edges,
    mode=cv2.RETR_EXTERNAL,       # only outermost contours
    method=cv2.CHAIN_APPROX_SIMPLE # compress straight segments
)

print(f"Found {len(contours)} contours")

# Filter by area (ignore tiny noise contours)
min_area = 500
large_contours = [c for c in contours if cv2.contourArea(c) > min_area]
print(f"Contours with area > {min_area}px: {len(large_contours)}")

# Draw contours and bounding boxes
result = scene.copy()
cv2.drawContours(result, large_contours, -1, (0, 255, 0), 2)

for c in large_contours:
    x, y, w_r, h_r = cv2.boundingRect(c)
    cv2.rectangle(result, (x, y), (x + w_r, y + h_r), (0, 0, 255), 1)
    area = cv2.contourArea(c)
    cv2.putText(result, f'{area:.0f}px', (x, y - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

show_images([
    (edges,  f'Canny edges'),
    (result, f'Contours (area > {min_area}) + bounding boxes'),
])

## 7. Histograms and Contrast Enhancement

### What a histogram tells you

A pixel value histogram shows the **distribution of intensities** in an image:
- Peaks at left → image is dark
- Peaks at right → image is bright
- Narrow distribution → low contrast
- Wide distribution → high contrast

**CLAHE** (Contrast Limited Adaptive Histogram Equalization) improves contrast locally — much better than global equalization for real-world images with uneven lighting.

In [ ]:
# Create a low-contrast version of the scene
dark_scene = (scene.astype(np.float32) * 0.4 + 30).clip(0, 255).astype(np.uint8)
gray_dark = cv2.cvtColor(dark_scene, cv2.COLOR_BGR2GRAY)

# Global histogram equalization
gray_eq = cv2.equalizeHist(gray_dark)

# CLAHE — contrast limited adaptive histogram equalization
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
gray_clahe = clahe.apply(gray_dark)

# Plot images and histograms
fig, axes = plt.subplots(3, 2, figsize=(12, 10))

for row, (img_show, title) in enumerate([
    (gray_dark,  'Dark original'),
    (gray_eq,    'Global equalization'),
    (gray_clahe, 'CLAHE (adaptive)'),
]):
    axes[row, 0].imshow(img_show, cmap='gray', vmin=0, vmax=255)
    axes[row, 0].set_title(title)
    axes[row, 0].axis('off')
    
    axes[row, 1].hist(img_show.ravel(), bins=256, range=(0, 256), color='steelblue', alpha=0.8)
    axes[row, 1].set_title(f'{title} — histogram')
    axes[row, 1].set_xlim(0, 255)
    axes[row, 1].set_ylabel('Pixel count')
    axes[row, 1].set_xlabel('Intensity')

plt.tight_layout()
plt.show()
print("CLAHE avoids the over-amplification artifacts of global equalization")

## 8. Practical Preprocessing Pipeline

In real robotics systems, you chain these operations into a preprocessing pipeline. Here is a typical pipeline for detecting a colored object in a noisy industrial camera feed:

In [ ]:
def preprocess_for_object_detection(frame_bgr, target_color='orange'):
    """
    Preprocessing pipeline for colored object detection.
    Returns: (binary mask, annotated frame, intermediate steps)
    """
    steps = {'original': frame_bgr.copy()}
    
    # Step 1: Reduce resolution for speed (if needed)
    h, w = frame_bgr.shape[:2]
    working = cv2.resize(frame_bgr, (w // 2, h // 2), interpolation=cv2.INTER_AREA)
    scale_factor = 2  # to map detections back to original size
    steps['resized'] = working.copy()
    
    # Step 2: Gaussian blur to reduce sensor noise
    blurred = cv2.GaussianBlur(working, (5, 5), 0)
    steps['blurred'] = blurred.copy()
    
    # Step 3: Convert to HSV for color segmentation
    hsv_img = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    
    # Step 4: HSV thresholding
    color_ranges = {
        'orange': (np.array([10, 100, 80]),  np.array([30, 255, 255])),
        'blue':   (np.array([100, 80, 80]),  np.array([130, 255, 255])),
        'green':  (np.array([35, 80, 80]),   np.array([85, 255, 255])),
    }
    lo, hi = color_ranges.get(target_color, color_ranges['orange'])
    mask = cv2.inRange(hsv_img, lo, hi)
    steps['raw_mask'] = mask.copy()
    
    # Step 5: Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)  # remove noise
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)  # fill gaps
    steps['clean_mask'] = mask.copy()
    
    # Step 6: Find contours and filter by area
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_area = 200  # pixels in working resolution
    valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]
    
    # Step 7: Annotate original frame
    annotated = frame_bgr.copy()
    for c in valid_contours:
        # Scale contour back to original resolution
        c_scaled = (c * scale_factor).astype(np.int32)
        x, y, wb, hb = cv2.boundingRect(c_scaled)
        area = cv2.contourArea(c_scaled)
        
        cv2.rectangle(annotated, (x, y), (x+wb, y+hb), (0, 255, 0), 2)
        cv2.putText(annotated, f'{target_color} ({area:.0f}px)',
                    (x, y - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
    
    return mask, annotated, steps

# Run the pipeline
mask, annotated, steps = preprocess_for_object_detection(scene, target_color='orange')

# Show all pipeline steps
show_images([
    (steps['original'],   '1. Original'),
    (steps['blurred'],    '2. Gaussian blur'),
    (steps['raw_mask'],   '3. HSV threshold'),
    (steps['clean_mask'], '4. Morphological cleanup'),
    (annotated,           '5. Final detection'),
], figsize_per=(4, 4))

print("Pipeline summary:")
print("  Resize → Blur → HSV threshold → Morph cleanup → Contours → Annotate")

## Recap

| Operation | Key function | Notes |
|---|---|---|
| Resize | `cv2.resize(img, (W, H), interpolation=...)` | Use `INTER_AREA` for shrinking |
| Crop | `img[y1:y2, x1:x2]` | NumPy slicing — rows first |
| Color convert | `cv2.cvtColor(img, cv2.COLOR_BGR2HSV)` | HSV best for color segmentation |
| Threshold | `cv2.threshold()` / `cv2.adaptiveThreshold()` | Otsu's for auto threshold |
| Morphology | `cv2.erode/dilate/morphologyEx` | Open=denoise, Close=fill |
| Blur | Gaussian/Median/Bilateral | Blur before edge detection |
| Edges | `cv2.Canny(img, low, high)` | Ratio 1:3 is a good starting point |
| Contrast | `cv2.createCLAHE().apply()` | Better than global equalization |
| Pipeline | Resize→Blur→HSV→Morph→Contours | Standard robotics workflow |

---

**Next:** `06_camera_model_theory.ipynb` (Part 3) — understand how a camera turns 3D world points into 2D pixels, and the math that makes 6D pose estimation possible.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Segment a colored object with HSV thresholding
# ============================================================
# The industrial scene has a blue-ish metal pallet.
# Task:
#   1. Load the industrial_scene.jpg image
#   2. Convert to HSV
#   3. Find appropriate HSV range for the dark blue/gray pallet
#      (hint: it has low saturation, mid value — check the HSV channels)
#   4. Create a mask
#   5. Apply morphological cleanup
#   6. Display: original | HSV-S channel | mask | segmented result
#
# Tip: low saturation objects (gray/metal) are best detected with
# combined S and V thresholds rather than H alone.

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# img = cv2.imread('../assets/images/industrial_scene.jpg')
# hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
#
# # Metal pallet: dark blue-gray, low saturation, medium value
# lower = np.array([90, 20, 40])
# upper = np.array([130, 120, 130])
# mask = cv2.inRange(hsv, lower, upper)
#
# # Morphological cleanup
# kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
# mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
# mask_clean = cv2.morphologyEx(mask_clean, cv2.MORPH_CLOSE, kernel)
#
# segmented = cv2.bitwise_and(img, img, mask=mask_clean)
#
# show_images([
#     (img,            'Original'),
#     (hsv[:,:,1],     'HSV: Saturation'),
#     (mask_clean,     'Cleaned mask'),
#     (segmented,      'Segmented pallet'),
# ])

In [ ]:
# ============================================================
# EXERCISE 2: Edge detection pipeline for an industrial part
# ============================================================
# Create a preprocessing function that:
#   1. Takes a BGR image
#   2. Converts to grayscale
#   3. Applies CLAHE (for contrast enhancement)
#   4. Blurs (choose appropriate kernel size)
#   5. Runs Canny edge detection
#   6. Finds and filters contours by area (min_area parameter)
#   7. Returns: edges image, list of (contour, bounding_rect, area)
#
# Apply to the industrial scene and draw the 5 largest objects found.

# YOUR CODE HERE
def detect_edges_and_objects(image_bgr, min_area=1000, canny_low=30, canny_high=100):
    pass


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def detect_edges_and_objects(image_bgr, min_area=1000, canny_low=30, canny_high=100):
#     gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
#
#     # Contrast enhancement
#     clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
#     enhanced = clahe.apply(gray)
#
#     # Blur
#     blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)
#
#     # Canny edges
#     edges = cv2.Canny(blurred, canny_low, canny_high)
#
#     # Contours
#     contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#     results = []
#     for c in contours:
#         area = cv2.contourArea(c)
#         if area >= min_area:
#             rect = cv2.boundingRect(c)
#             results.append((c, rect, area))
#
#     results.sort(key=lambda x: x[2], reverse=True)  # largest first
#     return edges, results
#
# img = cv2.imread('../assets/images/industrial_scene.jpg')
# edges, objects = detect_edges_and_objects(img, min_area=500)
# print(f"Found {len(objects)} objects")
#
# result = img.copy()
# colors = [(0,255,0),(0,0,255),(255,0,0),(0,255,255),(255,0,255)]
# for i, (c, (x,y,w,h), area) in enumerate(objects[:5]):
#     col = colors[i % len(colors)]
#     cv2.rectangle(result, (x,y), (x+w,y+h), col, 2)
#     cv2.putText(result, f'#{i+1} {area:.0f}px', (x, y-5),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
#
# show_images([(edges, 'Canny edges'), (result, '5 largest objects')])